# S4 — AI Architect: Ejercicios de código

**Módulo:** Semester 4 — AI Architect  
**Lecciones relacionadas:** 4.2 Dependency Graphs · 4.4 Workflow Design · 4.6 Concurrencia · 4.8 Diseñar para el fallo

## Objetivo

Implementar desde cero las cuatro piezas de infraestructura que un AI Architect necesita para construir sistemas multi-agente robustos en producción:

1. **DAGExecutor** — ejecuta nodos en orden topológico con paralelismo real
2. **DynamicToolSelector** — nivel 2 del espectro: el LLM elige la herramienta
3. **MultiAgentOrchestrator** — fork-join y fan-out con asyncio
4. **CircuitBreaker** — protección ante fallos en cascada

## Instrucciones de uso

- Ejecuta las celdas en orden.
- En cada ejercicio: lee el enunciado, implementa en la celda de starter code, ejecuta los tests.
- Los tests **no requieren API key real**: usan mocks o valores locales.
- Si todos los `assert` pasan sin excepción, el ejercicio está correcto.
- Para el Ejercicio 2 puedes configurar `OPENAI_API_KEY` en entorno si quieres probar con la API real.


In [ ]:
# Dependencias
# %pip install openai asyncio typing-extensions  # ya incluidas en Python 3.11+

import asyncio
import time
import json
import os
from typing import Any, Callable, Dict, List, Optional
from unittest.mock import AsyncMock, MagicMock, patch

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'sk-test')

print('Entorno listo.')


## Ejercicio 1 — DAGExecutor

**Contexto (lección 4.2):** El dependency graph es el primer artefacto de diseño de cualquier sistema multi-agente. Los nodos sin dependencias entre sí deben ejecutarse en paralelo — siempre. Hacerlos secuenciales añade latencia sin razón.

**Qué implementar:**

- `DAGExecutor.add_node(name, func, dependencies=[])` — registra un nodo con su función async y lista de dependencias.
- `DAGExecutor.execute()` — ejecuta el DAG respetando dependencias, con paralelismo real (asyncio) para nodos independientes. Devuelve `dict[str, Any]` con los resultados de cada nodo.
- `CyclicDependencyError` — excepción lanzada si el grafo tiene un ciclo.

**Criterios de éxito:**
- DAG de 5 nodos con 2 paralelos: los paralelos terminan antes que la suma de sus tiempos individuales.
- DAG con ciclo A→B→A lanza `CyclicDependencyError`.
- Cada nodo recibe como argumento el dict de resultados de sus dependencias.


In [ ]:
class CyclicDependencyError(Exception):
    """El grafo de dependencias contiene un ciclo — fallo de diseño."""
    pass


class DAGExecutor:
    """
    Ejecuta un grafo acíclico dirigido (DAG) de nodos async,
    paralelizando los nodos sin dependencias entre sí.
    """

    def __init__(self):
        # TODO: inicializa las estructuras de datos necesarias
        # Necesitas almacenar: nombre → función, nombre → dependencias
        raise NotImplementedError

    def add_node(
        self,
        name: str,
        func: Callable,
        dependencies: List[str] = None
    ) -> None:
        """
        Registra un nodo en el grafo.

        Args:
            name: identificador único del nodo
            func: función async que recibe dict[dep_name → result] y devuelve su resultado
            dependencies: lista de nombres de nodos cuyo resultado este nodo necesita
        """
        # TODO: almacena el nodo
        raise NotImplementedError

    def _topological_order(self) -> List[List[str]]:
        """
        Devuelve listas de nodos agrupados por nivel de ejecución.
        Nodos del mismo nivel pueden ejecutarse en paralelo.
        Lanza CyclicDependencyError si hay un ciclo (Kahn's algorithm).

        Ejemplo de output para A→C, B→C, C→D:
            [["A", "B"], ["C"], ["D"]]
        """
        # TODO: implementa Kahn's algorithm (BFS topológico)
        # 1. Calcula in-degree de cada nodo
        # 2. Añade a la cola los nodos con in-degree 0
        # 3. En cada iteración: procesa todos los nodos de la cola (= un nivel paralelo)
        #    y reduce el in-degree de sus dependientes
        # 4. Si al final quedan nodos sin procesar → hay un ciclo
        raise NotImplementedError

    async def execute(self) -> Dict[str, Any]:
        """
        Ejecuta el DAG completo y devuelve {node_name: result}.
        Paraleliza los nodos de cada nivel con asyncio.gather.
        """
        # TODO:
        # 1. Obtén el orden topológico por niveles
        # 2. Para cada nivel, lanza todos los nodos en paralelo con asyncio.gather
        # 3. Cada nodo recibe un dict con los resultados de sus dependencias
        # 4. Acumula los resultados y devuelve el dict completo
        raise NotImplementedError


In [ ]:
# Tests Ejercicio 1

async def _test_dag():
    execution_log = []  # registra el orden real de ejecución

    # Grafo: busqueda_externa y busqueda_interna son paralelos
    # input → busqueda_externa ↘
    #       → busqueda_interna ↗ → analisis → redaccion → formato

    async def nodo_input(deps):
        execution_log.append('input')
        return {'empresa': 'Acme Corp'}

    async def nodo_busqueda_externa(deps):
        await asyncio.sleep(0.08)  # simula latencia
        execution_log.append('busqueda_externa')
        return {'fuentes': ['noticia1', 'noticia2']}

    async def nodo_busqueda_interna(deps):
        await asyncio.sleep(0.06)  # más rápido que externa
        execution_log.append('busqueda_interna')
        return {'historico': ['informe_Q3']}

    async def nodo_analisis(deps):
        # Debe recibir resultados de ambas búsquedas
        assert 'busqueda_externa' in deps
        assert 'busqueda_interna' in deps
        execution_log.append('analisis')
        return {'analisis': 'SWOT completado'}

    async def nodo_formato(deps):
        assert 'analisis' in deps
        execution_log.append('formato')
        return {'pdf': 'informe_final.pdf'}

    dag = DAGExecutor()
    dag.add_node('input', nodo_input, [])
    dag.add_node('busqueda_externa', nodo_busqueda_externa, ['input'])
    dag.add_node('busqueda_interna', nodo_busqueda_interna, ['input'])
    dag.add_node('analisis', nodo_analisis, ['busqueda_externa', 'busqueda_interna'])
    dag.add_node('formato', nodo_formato, ['analisis'])

    t0 = time.perf_counter()
    results = await dag.execute()
    elapsed = time.perf_counter() - t0

    # Verifica resultados
    assert 'input' in results
    assert 'busqueda_externa' in results
    assert 'busqueda_interna' in results
    assert 'analisis' in results
    assert 'formato' in results
    assert results['formato']['pdf'] == 'informe_final.pdf'

    # Verifica paralelismo: las dos búsquedas paralelas tardan ~0.08s (max), no 0.14s (sum)
    assert elapsed < 0.25, f'Demasiado lento ({elapsed:.2f}s) — ¿están paralelizando?'

    # Verifica orden: analisis y formato van después de las búsquedas
    assert execution_log.index('analisis') > execution_log.index('busqueda_externa')
    assert execution_log.index('analisis') > execution_log.index('busqueda_interna')
    assert execution_log.index('formato') > execution_log.index('analisis')

    print(f'DAG ejecutado en {elapsed:.3f}s | orden: {execution_log}')

    # Verifica detección de ciclos
    dag_ciclico = DAGExecutor()

    async def dummy(deps): return {}

    dag_ciclico.add_node('A', dummy, ['B'])
    dag_ciclico.add_node('B', dummy, ['C'])
    dag_ciclico.add_node('C', dummy, ['A'])  # ciclo A→B→C→A

    try:
        await dag_ciclico.execute()
        assert False, 'Debería haber lanzado CyclicDependencyError'
    except CyclicDependencyError:
        print('CyclicDependencyError correctamente detectado')

    print('✅ Ejercicio 1 OK')


asyncio.run(_test_dag())


## Ejercicio 2 — DynamicToolSelector (nivel 2 del espectro)

**Contexto (lección 4.4):** El nivel 2 del espectro determinista-dinámico es: el LLM elige entre herramientas finitas, pero el código ejecuta. Es la zona de producción robusta — flexibilidad controlada sin que el LLM pueda hacer cosas no previstas.

**Qué implementar:**

- `DynamicToolSelector(tools: dict)` — recibe un diccionario `{nombre: {description, func}}`.
- `select_and_execute(query: str) -> dict` — construye un prompt con la query y las herramientas disponibles, pide al LLM que devuelva el nombre de la herramienta a usar (JSON con campo `tool`), ejecuta esa herramienta y devuelve `{tool_used, result}`.
- `_build_prompt(query)` — genera el prompt de selección (método separado, testeable sin API).
- `_parse_tool_choice(response: str) -> str` — parsea la respuesta del LLM, robusto ante whitespace y mayúsculas.

**Criterios de éxito:**
- El prompt incluye la descripción de todas las herramientas disponibles.
- Con mock del LLM: 3 queries → 3 herramientas correctas seleccionadas.
- `_parse_tool_choice` maneja JSON con espacios, comillas simples y nombre en cualquier capitalización.


In [ ]:
class ToolNotFoundError(Exception):
    """El LLM eligió una herramienta que no existe en el registry."""
    pass


class DynamicToolSelector:
    """
    Nivel 2 del espectro de dinamismo (lección 4.4):
    el LLM elige herramienta, el código la ejecuta.
    """

    def __init__(self, tools: Dict[str, Dict]):
        """
        Args:
            tools: {nombre: {"description": str, "func": callable}}
        """
        # TODO: almacena las herramientas
        raise NotImplementedError

    def _build_prompt(self, query: str) -> str:
        """
        Construye el prompt de selección de herramienta.
        Debe incluir:
        - La query del usuario
        - La lista de herramientas disponibles con sus descripciones
        - Instrucción de devolver JSON: {"tool": "nombre_herramienta"}
        - Solo nombres válidos (los del registry)
        """
        # TODO
        raise NotImplementedError

    def _parse_tool_choice(self, response: str) -> str:
        """
        Parsea la respuesta del LLM y devuelve el nombre de la herramienta.
        Debe ser robusto: manejar whitespace extra, capitalización,
        y respuesta con markdown (```json ... ```).
        Lanza ToolNotFoundError si el nombre no existe en el registry.
        """
        # TODO: 
        # 1. Limpia la respuesta (strip, quita ```json si está)
        # 2. Parsea JSON
        # 3. Extrae campo "tool" (case-insensitive lookup en el registry)
        # 4. Verifica que existe en self.tools
        raise NotImplementedError

    def _call_llm(self, prompt: str) -> str:
        """
        Llama al LLM y devuelve la respuesta como string.
        Usa openai.OpenAI con el modelo gpt-4o-mini.
        """
        # TODO: llama a openai.OpenAI().chat.completions.create
        # model='gpt-4o-mini', messages=[{role:system, content:prompt}]
        # devuelve choices[0].message.content
        raise NotImplementedError

    def select_and_execute(self, query: str) -> Dict:
        """
        Pipeline completo: prompt → LLM → parse → execute.
        Devuelve: {"tool_used": str, "result": Any}
        """
        # TODO:
        # 1. _build_prompt(query)
        # 2. _call_llm(prompt)
        # 3. _parse_tool_choice(response)
        # 4. Ejecuta self.tools[tool_name]["func"](query)
        # 5. Devuelve {"tool_used": tool_name, "result": resultado}
        raise NotImplementedError


In [ ]:
# Tests Ejercicio 2 — usa mock del LLM, no requiere API key real

# Herramientas de ejemplo
tools = {
    'buscar_web': {
        'description': 'Busca información actualizada en internet. Úsala para noticias, datos recientes o información que cambia.',
        'func': lambda q: {'resultado': f'Resultados web para: {q}', 'fuentes': 3}
    },
    'consultar_bd': {
        'description': 'Consulta la base de datos interna de clientes y pedidos. Úsala para datos de clientes específicos.',
        'func': lambda q: {'cliente': 'Acme', 'pedidos': 42}
    },
    'calcular': {
        'description': 'Realiza cálculos matemáticos o financieros. Úsala para sumas, porcentajes, proyecciones.',
        'func': lambda q: {'resultado': 1337, 'operacion': 'completada'}
    }
}

selector = DynamicToolSelector(tools)

# Test 1: el prompt incluye todas las herramientas
prompt = selector._build_prompt('¿Cuántos pedidos tiene el cliente Acme?')
assert 'buscar_web' in prompt, 'El prompt debe listar todas las herramientas'
assert 'consultar_bd' in prompt
assert 'calcular' in prompt
assert 'Acme' in prompt, 'El prompt debe incluir la query del usuario'
assert '{"tool"' in prompt or "'tool'" in prompt or 'tool' in prompt.lower(), 'El prompt debe indicar el formato de respuesta'
print('✅ _build_prompt OK')

# Test 2: _parse_tool_choice maneja variantes de respuesta
casos = [
    ('{"tool": "consultar_bd"}', 'consultar_bd'),
    ('  { "tool" : "buscar_web" }  ', 'buscar_web'),
    ('```json\n{"tool": "calcular"}\n```', 'calcular'),
    ('{"tool": "CONSULTAR_BD"}', 'consultar_bd'),  # case-insensitive
]
for respuesta, esperado in casos:
    resultado = selector._parse_tool_choice(respuesta)
    assert resultado == esperado, f'Para "{respuesta}" esperaba "{esperado}", obtuve "{resultado}"'
print('✅ _parse_tool_choice OK')

# Test 3: herramienta inválida lanza ToolNotFoundError
try:
    selector._parse_tool_choice('{"tool": "herramienta_inexistente"}')
    assert False, 'Debería lanzar ToolNotFoundError'
except ToolNotFoundError:
    print('✅ ToolNotFoundError correctamente lanzado')

# Test 4: select_and_execute con mock del LLM
queries_esperadas = [
    ('¿Cuántos pedidos tiene el cliente Acme?', 'consultar_bd'),
    ('Busca noticias sobre OpenAI de esta semana', 'buscar_web'),
    ('Calcula el 15% de 8500 euros', 'calcular'),
]

for query, tool_esperada in queries_esperadas:
    # Mock: el LLM devuelve siempre la herramienta correcta
    respuesta_mock = json.dumps({'tool': tool_esperada})
    with patch.object(selector, '_call_llm', return_value=respuesta_mock):
        resultado = selector.select_and_execute(query)
    assert resultado['tool_used'] == tool_esperada, f'Para "{query}" esperaba {tool_esperada}'
    assert 'result' in resultado
    print(f'  query: "{query[:40]}..." → tool: {resultado["tool_used"]} ✅')

print('✅ Ejercicio 2 OK')


## Ejercicio 3 — MultiAgentOrchestrator (fork-join paralelo)

**Contexto (lección 4.6):** En el patrón fork-join el orquestador lanza N agentes en paralelo y espera a que todos terminen. El tiempo total es `max(agentes)`, no `sum(agentes)`. La diferencia entre secuencial y paralelo es la palanca de latencia más directa que tienes.

**Qué implementar:**

- `MultiAgentOrchestrator(agents: dict)` — diccionario `{nombre: función_async}`.
- `run_parallel(tasks: list[dict]) -> list[dict]` — ejecuta `[{agent, input}]` con `asyncio.gather`. Devuelve `[{agent, input, output, duration_ms}]`.
- `run_sequential(tasks: list[dict]) -> list[dict]` — misma interfaz, ejecución secuencial.
- Manejo de errores: si un agente lanza excepción, el resultado tiene `{error: str}` en lugar de `output`, sin detener los demás.

**Criterios de éxito:**
- `run_parallel` con 3 agentes de 0.1s tarda < 0.2s (no 0.3s).
- `run_sequential` con los mismos tarda > 0.25s.
- Un agente que falla no rompe los demás en `run_parallel`.
- Todos los campos `{agent, input, output, duration_ms}` presentes en cada resultado.


In [ ]:
class MultiAgentOrchestrator:
    """
    Orquestador fork-join: lanza agentes en paralelo y espera a todos.
    Lección 4.6 — patrón fork-join.
    """

    def __init__(self, agents: Dict[str, Callable]):
        """
        Args:
            agents: {nombre_agente: función_async(input) -> output}
        """
        # TODO
        raise NotImplementedError

    async def _run_one(
        self,
        agent_name: str,
        task_input: Any
    ) -> Dict:
        """
        Ejecuta un agente individual. Captura excepciones.
        Devuelve: {agent, input, output, duration_ms}
        Si hay excepción: {agent, input, error, duration_ms}
        """
        # TODO:
        # 1. Registra t0 = time.perf_counter()
        # 2. Llama al agente con task_input (captura Exception)
        # 3. Calcula duration_ms = (time.perf_counter() - t0) * 1000
        # 4. Devuelve el dict con los campos requeridos
        raise NotImplementedError

    async def run_parallel(self, tasks: List[Dict]) -> List[Dict]:
        """
        Ejecuta todas las tasks en paralelo con asyncio.gather.

        Args:
            tasks: [{"agent": str, "input": Any}, ...]

        Returns:
            [{"agent", "input", "output", "duration_ms"}, ...] en el mismo orden que tasks
        """
        # TODO: usa asyncio.gather con return_exceptions=False (ya gestionamos en _run_one)
        raise NotImplementedError

    async def run_sequential(self, tasks: List[Dict]) -> List[Dict]:
        """
        Ejecuta las tasks una a una. Misma interfaz que run_parallel.
        Sirve como línea de referencia para comparar latencia.
        """
        # TODO: itera sobre tasks y llama _run_one secuencialmente
        raise NotImplementedError


In [ ]:
# Tests Ejercicio 3

async def _test_orchestrator():

    # Agentes simulados con latencia fija
    async def agente_analisis(input_data):
        await asyncio.sleep(0.10)
        return {'analisis': f'analizado: {input_data}'}

    async def agente_resumen(input_data):
        await asyncio.sleep(0.10)
        return {'resumen': f'resumido: {input_data}'}

    async def agente_clasificador(input_data):
        await asyncio.sleep(0.10)
        return {'categoria': 'positivo'}

    async def agente_roto(input_data):
        await asyncio.sleep(0.05)
        raise ValueError('Error simulado de agente')

    orchestrator = MultiAgentOrchestrator({
        'analisis': agente_analisis,
        'resumen': agente_resumen,
        'clasificador': agente_clasificador,
        'roto': agente_roto,
    })

    tasks = [
        {'agent': 'analisis', 'input': 'texto de prueba A'},
        {'agent': 'resumen', 'input': 'texto de prueba B'},
        {'agent': 'clasificador', 'input': 'texto de prueba C'},
    ]

    # Test 1: run_parallel es más rápido que run_sequential
    t0 = time.perf_counter()
    results_parallel = await orchestrator.run_parallel(tasks)
    t_parallel = time.perf_counter() - t0

    t0 = time.perf_counter()
    results_sequential = await orchestrator.run_sequential(tasks)
    t_sequential = time.perf_counter() - t0

    print(f'Paralelo: {t_parallel:.3f}s | Secuencial: {t_sequential:.3f}s')
    assert t_parallel < 0.20, f'run_parallel demasiado lento: {t_parallel:.3f}s (esperado < 0.20s)'
    assert t_sequential > 0.25, f'run_sequential demasiado rápido: {t_sequential:.3f}s (esperado > 0.25s)'
    assert t_parallel < t_sequential, 'Paralelo debe ser más rápido que secuencial'
    print('✅ Paralelismo verificado')

    # Test 2: todos los campos presentes
    for r in results_parallel:
        assert 'agent' in r, f'Falta campo agent: {r}'
        assert 'input' in r, f'Falta campo input: {r}'
        assert 'output' in r or 'error' in r, f'Falta output o error: {r}'
        assert 'duration_ms' in r, f'Falta duration_ms: {r}'
        assert r['duration_ms'] > 0
    print('✅ Campos de resultado correctos')

    # Test 3: orden preservado
    assert results_parallel[0]['agent'] == 'analisis'
    assert results_parallel[1]['agent'] == 'resumen'
    assert results_parallel[2]['agent'] == 'clasificador'
    print('✅ Orden de resultados preservado')

    # Test 4: agente roto no rompe los demás
    tasks_con_fallo = [
        {'agent': 'analisis', 'input': 'texto'},
        {'agent': 'roto', 'input': 'texto'},
        {'agent': 'resumen', 'input': 'texto'},
    ]
    results_fallo = await orchestrator.run_parallel(tasks_con_fallo)
    assert results_fallo[0].get('output') is not None, 'analisis debe tener output'
    assert 'error' in results_fallo[1], 'roto debe tener campo error'
    assert results_fallo[2].get('output') is not None, 'resumen debe tener output'
    print(f'✅ Fallo aislado: {results_fallo[1]["error"]}')

    print('✅ Ejercicio 3 OK')


asyncio.run(_test_orchestrator())


## Ejercicio 4 — CircuitBreaker

**Contexto (lección 4.8):** Los agentes van a fallar. El circuit breaker es el mecanismo que previene que un agente en estado de fallo sistémico consuma tokens en reintentos condenados al fracaso. Tres estados: CLOSED (funciona), OPEN (falla rápido), HALF_OPEN (sondeo de recuperación).

**Qué implementar:**

- `CircuitBreaker(failure_threshold=3, reset_timeout=60)`
- `call(func, *args, **kwargs)` — ejecuta `func` protegida por el circuit breaker.
- Estados: `CLOSED` → `OPEN` tras `failure_threshold` fallos consecutivos. `OPEN` → `HALF_OPEN` tras `reset_timeout` segundos. `HALF_OPEN` → `CLOSED` si la llamada tiene éxito; → `OPEN` si falla.
- En estado `OPEN`: lanza `CircuitOpenError` sin llamar a `func`.

**Criterios de éxito:**
- Cada transición de estado verificada con función que falla/tiene éxito de forma controlada.
- En OPEN, `func` nunca se llama (verificado con contador de llamadas).
- `reset_timeout` probado con tiempo simulado (monkeypatching de `time.time`).


In [ ]:
from enum import Enum


class CircuitState(Enum):
    CLOSED = 'CLOSED'        # funcionando — deja pasar llamadas
    OPEN = 'OPEN'            # fallando — rechaza llamadas sin ejecutar
    HALF_OPEN = 'HALF_OPEN'  # sondeo — deja pasar UNA llamada de prueba


class CircuitOpenError(Exception):
    """Lanzada cuando el circuit está OPEN y se intenta hacer una llamada."""
    pass


class CircuitBreaker:
    """
    Protege llamadas a funciones (agentes, APIs) ante fallos sistémicos.
    Lección 4.8 — diseñar para el fallo.

    Estados:
        CLOSED   → llamadas normales. Tras failure_threshold fallos → OPEN.
        OPEN     → falla rápido (CircuitOpenError). Tras reset_timeout s → HALF_OPEN.
        HALF_OPEN → permite UNA llamada. Éxito → CLOSED. Fallo → OPEN.
    """

    def __init__(self, failure_threshold: int = 3, reset_timeout: float = 60.0):
        """
        Args:
            failure_threshold: fallos consecutivos para abrir el circuito
            reset_timeout: segundos en OPEN antes de pasar a HALF_OPEN
        """
        # TODO: inicializa estado, contador de fallos, timestamp de apertura
        raise NotImplementedError

    @property
    def state(self) -> CircuitState:
        """Devuelve el estado actual, evaluando si OPEN debe pasar a HALF_OPEN."""
        # TODO:
        # Si el estado interno es OPEN y han pasado >= reset_timeout segundos
        # desde que se abrió, devuelve HALF_OPEN (pero no cambia el estado interno todavía)
        raise NotImplementedError

    def call(self, func: Callable, *args, **kwargs) -> Any:
        """
        Ejecuta func protegida por el circuit breaker.

        Raises:
            CircuitOpenError: si el estado es OPEN
            Exception: si func lanza excepción (y gestiona la transición de estado)
        """
        # TODO:
        # 1. Lee self.state (property que evalúa transición OPEN→HALF_OPEN)
        # 2. Si OPEN: lanza CircuitOpenError
        # 3. Si CLOSED o HALF_OPEN: intenta llamar a func(*args, **kwargs)
        #    - Éxito:
        #        * Resetea consecutive_failures a 0
        #        * Si venía de HALF_OPEN → CLOSED
        #        * Devuelve resultado
        #    - Fallo:
        #        * Incrementa consecutive_failures
        #        * Si HALF_OPEN → OPEN inmediatamente (con nuevo timestamp)
        #        * Si CLOSED y consecutive_failures >= threshold → OPEN (con timestamp)
        #        * Re-lanza la excepción original
        raise NotImplementedError


In [ ]:
# Tests Ejercicio 4

# --- Test 1: CLOSED → OPEN tras failure_threshold fallos ---
cb = CircuitBreaker(failure_threshold=3, reset_timeout=60)
assert cb.state == CircuitState.CLOSED

call_count = 0

def func_que_falla():
    global call_count
    call_count += 1
    raise ConnectionError('timeout del agente')

# 2 fallos → sigue CLOSED
for _ in range(2):
    try:
        cb.call(func_que_falla)
    except ConnectionError:
        pass

assert cb.state == CircuitState.CLOSED, f'Después de 2 fallos debe seguir CLOSED, está {cb.state}'
print('✅ 2 fallos → sigue CLOSED')

# 3er fallo → OPEN
try:
    cb.call(func_que_falla)
except ConnectionError:
    pass

assert cb.state == CircuitState.OPEN, f'Después de 3 fallos debe estar OPEN, está {cb.state}'
print('✅ 3 fallos → OPEN')

# --- Test 2: OPEN falla rápido sin llamar a func ---
call_count_antes = call_count
try:
    cb.call(func_que_falla)
    assert False, 'Debería lanzar CircuitOpenError'
except CircuitOpenError:
    pass

assert call_count == call_count_antes, 'En OPEN, func NO debe ser llamada'
print('✅ OPEN → CircuitOpenError sin llamar a func')

# --- Test 3: OPEN → HALF_OPEN tras reset_timeout ---
# Monkeypatch de time.time para simular el paso del tiempo
cb2 = CircuitBreaker(failure_threshold=2, reset_timeout=30)

def func_falla_siempre():
    raise RuntimeError('fallo')

for _ in range(2):
    try:
        cb2.call(func_falla_siempre)
    except RuntimeError:
        pass

assert cb2.state == CircuitState.OPEN

# Simula que pasaron 31 segundos
with patch('time.time', return_value=time.time() + 31):
    assert cb2.state == CircuitState.HALF_OPEN, f'Tras timeout debería ser HALF_OPEN, es {cb2.state}'
print('✅ OPEN → HALF_OPEN tras reset_timeout')

# --- Test 4: HALF_OPEN → CLOSED si éxito ---
cb3 = CircuitBreaker(failure_threshold=2, reset_timeout=0)  # timeout=0 → pasa a HALF_OPEN inmediatamente

for _ in range(2):
    try:
        cb3.call(func_falla_siempre)
    except RuntimeError:
        pass

assert cb3.state in (CircuitState.OPEN, CircuitState.HALF_OPEN)  # con timeout=0 puede ser cualquiera

# Fuerza HALF_OPEN
with patch('time.time', return_value=time.time() + 1):
    assert cb3.state == CircuitState.HALF_OPEN

    def func_exitosa():
        return 'ok'

    resultado = cb3.call(func_exitosa)
    assert resultado == 'ok'
    assert cb3.state == CircuitState.CLOSED, f'Tras éxito en HALF_OPEN debería ser CLOSED, es {cb3.state}'

print('✅ HALF_OPEN + éxito → CLOSED')

# --- Test 5: HALF_OPEN → OPEN si fallo ---
cb4 = CircuitBreaker(failure_threshold=2, reset_timeout=0)

for _ in range(2):
    try:
        cb4.call(func_falla_siempre)
    except RuntimeError:
        pass

with patch('time.time', return_value=time.time() + 1):
    assert cb4.state == CircuitState.HALF_OPEN
    try:
        cb4.call(func_falla_siempre)
    except RuntimeError:
        pass
    # Después del fallo en HALF_OPEN volvemos a OPEN
    # (sin sumar más tiempo, el estado interno es OPEN)
    assert cb4._state == CircuitState.OPEN, f'Tras fallo en HALF_OPEN debería ser OPEN, es {cb4._state}'

print('✅ HALF_OPEN + fallo → OPEN')
print('✅ Ejercicio 4 OK')


## Tips y referencias

### Ejercicio 1 — DAGExecutor
- **Algoritmo de Kahn**: mantén un `in_degree` dict y una cola de nodos con grado 0. En cada iteración, procesa todos los nodos de la cola en paralelo (son el mismo "nivel"). Reduce el `in_degree` de sus dependientes. Si al final quedan nodos sin procesar, hay un ciclo.
- `asyncio.gather(*coroutines)` ejecuta todas en paralelo y devuelve resultados en el mismo orden.
- 📖 Lección 4.2 — Dependency Graphs, Bloque 2: Backward chaining

### Ejercicio 2 — DynamicToolSelector
- El prompt debe ser específico: lista las herramientas con nombre y descripción, pide exactamente `{"tool": "nombre"}`, incluye solo los nombres válidos.
- Para `_parse_tool_choice`: usa `response.strip()`, luego elimina fences de markdown con regex, luego `json.loads()`.
- 📖 Lección 4.4 — Workflow Design, Bloque 2: Los cinco niveles en detalle (nivel 2)

### Ejercicio 3 — MultiAgentOrchestrator
- `asyncio.gather(*tasks, return_exceptions=True)` — pero mejor gestionar excepciones dentro de `_run_one` para control granular.
- El tiempo de `run_parallel` debe ser ≈ `max(durations)`, no `sum(durations)`.
- 📖 Lección 4.6 — Concurrencia, Bloque 3: Los patrones de coordinación (fork-join)

### Ejercicio 4 — CircuitBreaker
- El atributo interno de estado llámalo `_state` (no `state`, que es la property).
- La property `state` evalúa en tiempo real si hay que pasar de OPEN a HALF_OPEN comparando `time.time()` con el timestamp de apertura.
- En HALF_OPEN solo permites UNA llamada de prueba — si falla, vuelves a OPEN con nuevo timestamp.
- 📖 Lección 4.8 — Diseñar para el fallo, Bloque 2: Los tres niveles de resiliencia
